### **Component-wise analytical vRAM profile for `llava-hf/llava-1.5-7b-hf`**

In [1]:
import torch
import gc
from transformers import AutoProcessor, LlavaForConditionalGeneration
from PIL import Image
import requests
from io import BytesIO
import numpy as np

/home/eros483/miniconda3/envs/deepLure/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def clear_memory():
    """
    Helper Function.
    - Aggressively clear GPU memory and synchronize.
    """
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

In [3]:
def get_vram_usage():
    """
    Helper Function.
    - Returns the current VRAM usage in MB.
    """
    torch.cuda.synchronize()
    return torch.cuda.memory_allocated()/(1024**2)

In [4]:
def get_peak_vram_usage():
    """
    Helper Function.
    - Returns the peak VRAM usage in MB.
    """
    torch.cuda.synchronize()
    return torch.cuda.max_memory_allocated()/(1024**2)

In [5]:
def measure_phase(phase_name, func, *args, **kwargs):
    """
    Helper Function.
    - Wrapper to measure VRAM for a specific phase.
    """
    clear_memory()
    torch.cuda.reset_peak_memory_stats()
    start_mem = get_vram_usage()
    
    result = func(*args, **kwargs)
    
    torch.cuda.synchronize()
    end_mem = get_vram_usage()
    peak_mem = get_peak_vram_usage()
    
    phase_mem = end_mem - start_mem
    peak_delta = peak_mem - start_mem
    
    print(f"{phase_name} Net: {phase_mem:>8.2f} MB, Peak: {peak_delta:>8.2f} MB")
    return result, phase_mem, peak_delta

In [ ]:
model_id="llava-hf/llava-1.5-7b-hf"

- **Base memory Utilisation**

In [7]:
clear_memory()
base_mem = get_vram_usage()
print(f"Baseline VRAM Usage: {base_mem:.4f} MB")
print()

Baseline VRAM Usage: 0.0000 MB



- **Static Model Loading**

In [8]:
def load_model():
    model=LlavaForConditionalGeneration.from_pretrained(
            model_id, 
            torch_dtype=torch.float16, 
            device_map="cuda"
        )
    return model

model, model_mem, _= measure_phase("Model Loading", load_model)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 3/3 [00:00<00:00,  4.17it/s]


Model Loading Net: 13472.42 MB, Peak: 13472.42 MB


Processor is loaded in CPU, and does not utilise vram

In [9]:
def load_processor():
    processor=AutoProcessor.from_pretrained(model_id)
    return processor

processor, proc_mem, _= measure_phase("Processor Loading", load_processor)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Processor Loading Net:     0.00 MB, Peak:     0.00 MB


- **Image preprocessing and loading**

In [10]:
def load_test_image():
    #sample image
    url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/car.jpg"
    response = requests.get(url, timeout=10)
    image = Image.open(BytesIO(response.content))
    return image

In [11]:
image=load_test_image()
prompt="USER: <image>\nDescribe this image. ASSISTANT:"

In [12]:
def preprocess_on_cpu():
    """
    Process image and text on CPU.
    """
    return processor(text=prompt, images=image, return_tensors="pt")

inputs_cpu, preprocess_cpu_mem, _ = measure_phase(
    "Preprocessing on CPU", 
    preprocess_on_cpu
)

def transfer_to_gpu():
    """
    Transfer preprocessed tensors to GPU.
    """
    return inputs_cpu.to("cuda", torch.float16)

inputs, transfer_mem, _ = measure_phase("CPU→GPU Transfer", transfer_to_gpu)

Preprocessing on CPU Net:     0.00 MB, Peak:     0.00 MB
CPU→GPU Transfer Net:     0.66 MB, Peak:     0.66 MB


- **Vision Encoder Forward pass**

In [13]:
def vision_forward():
    """
    Run vision encoder forward pass for CLIP ViT.
    """
    with torch.no_grad():
        vision_outputs = model.vision_tower(
            inputs["pixel_values"],
            output_hidden_states=True
        )
        return vision_outputs.last_hidden_state

image_features, vision_mem, vision_peak_mem= measure_phase(
    "Vision Encoder Activations",
    vision_forward
)

feature_mem = (image_features.element_size() * image_features.nelement()) / (1024**2)
print(f"Image features tensor: {feature_mem:.2f} MB")
print(f"Activations: {vision_peak_mem-vision_mem:.2f} MB") #freed after forward

Vision Encoder Activations Net:    10.25 MB, Peak:    53.25 MB
Image features tensor: 1.13 MB
Activations: 43.00 MB


- **MLP projection**

In [14]:
def projection_forward():
    """
    Run MLP for forward pass.
    """
    with torch.no_grad():
        return model.multi_modal_projector(image_features)

projected_features, projector_mem, projector_peak_mem = measure_phase(
    "MLP Projection Layer",
    projection_forward
)

MLP Projection Layer Net:     4.51 MB, Peak:     9.02 MB


- **LLM Text Embedding and concatenation**

In [15]:
def text_embedding():
    """
    Embed text tokens.
    """
    with torch.no_grad():
        text_embeds = model.language_model.get_input_embeddings()(inputs["input_ids"])
        return text_embeds

text_embeds, text_embed_mem, text_embed_peak_mem = measure_phase(
    "Text Token Embedding",
    text_embedding
)

Text Token Embedding Net:     4.63 MB, Peak:     4.63 MB


In [16]:
def combine_embeddings():
    """
    Concatenate image and text embeddings.
    """
    with torch.no_grad():
        return torch.cat([projected_features, text_embeds], dim=1)

combined_embeds, combine_mem, combine_peak_mem = measure_phase(
    "Image+Text Concatenation",
    combine_embeddings
)

Image+Text Concatenation Net:    10.02 MB, Peak:    10.02 MB


In [17]:
combined_mem = (combined_embeds.element_size() * combined_embeds.nelement()) / (1024**2)
print(f"Combined embeddings VRAM Usage: {combined_mem:.2f} MB")
print()

Combined embeddings VRAM Usage: 9.14 MB



- **LLM Generation and Initial KV Cache**

In [18]:
def prefill_pass():
    """
    Initial forward pass through LLM\.
    """
    with torch.no_grad():
        return model(**inputs, use_cache=True)

outputs, prefill_mem, prefill_peak_mem = measure_phase(
    "Prefill Phase (Full Prompt)",
    prefill_pass
)

def get_kv_cache_size(past_key_values):
    """
    Calculate VRAM of KV-cache tensors for first LLM Pass.
    """
    if past_key_values is None:
        return 0.0
    total_elements = sum(t.numel() for layer in past_key_values for t in layer)
    return (total_elements * 2) / (1024**2)

initial_kv_mem = get_kv_cache_size(outputs.past_key_values)
KV_cache_activations=prefill_peak_mem-prefill_mem #dumped after pass
print(f"Initial KV-cache VRAM Usage: {initial_kv_mem:.2f} MB")
print(f"KV-cache Activations: {KV_cache_activations:.2f} MB")

Prefill Phase (Full Prompt) Net:   338.99 MB, Peak:   359.11 MB
Initial KV-cache VRAM Usage: 296.50 MB
KV-cache Activations: 20.12 MB


- **Decode Phase with KV Cache Growth**

In [19]:
print("-" * 70)
print(f"{'Step':<8} | {'KV-Cache (MB)':<15} | {'Growth (MB)':<12} | {'Total GPU (MB)':<15}")
print("-" * 70)

past_key_values = outputs.past_key_values
input_ids = outputs.logits[:, -1, :].argmax(-1).unsqueeze(-1)

prev_kv = initial_kv_mem
decode_mems = []

for i in range(1, 26): #running for 25 tokens
    clear_memory()
    start_total = get_vram_usage()
    
    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            past_key_values=past_key_values,
            use_cache=True
        )
    
    past_key_values = outputs.past_key_values
    input_ids = outputs.logits[:, -1, :].argmax(-1).unsqueeze(-1)
    
    curr_kv = get_kv_cache_size(past_key_values)
    total_gpu = get_vram_usage()
    growth = curr_kv - prev_kv
    
    decode_mems.append({
        'step': i,
        'kv_cache': curr_kv,
        'growth': growth,
        'total': total_gpu
    })
    
    # Log every 5 steps, first step, and last step
    if i % 5 == 0 or i == 1 or i == 25:
        print(f"{i:<8} | {curr_kv:<15.2f} | {growth:<12.4f} | {total_gpu:<15.2f}")
    
    prev_kv = curr_kv

----------------------------------------------------------------------
Step     | KV-Cache (MB)   | Growth (MB)  | Total GPU (MB) 
----------------------------------------------------------------------
1        | 297.00          | 0.5000       | 13801.29       
5        | 299.00          | 0.5000       | 13801.55       
10       | 301.50          | 0.5000       | 13804.05       
15       | 304.00          | 0.5000       | 13815.55       
20       | 306.50          | 0.5000       | 13818.35       
25       | 309.00          | 0.5000       | 13819.12       


- **Overall memory consumption breakdown**

In [20]:
print("Static Components:")
print(f"  Model Weights (FP16):        {model_mem:>10.2f} MB")
print()

print("Dynamic Components (per inference):")
print(f"  CPU→GPU Transfer:            {transfer_mem:>10.2f} MB\n")

print(f"  Vision Encoder (net):        {vision_mem:>10.2f} MB")
print(f"  Vision Encoder (peak):       {vision_peak_mem:>10.2f} MB\n")

print(f"  Projection Layer (net):      {projector_mem:>10.2f} MB")
print(f"  Projection Layer (peak):     {projector_peak_mem:>10.2f} MB\n")

print(f"  Text Embedding (net):        {text_embed_mem:>10.2f} MB")
print(f"  Text Embedding (peak):       {text_embed_peak_mem:>10.2f} MB\n")

print(f"  Embedding Fusion (net):      {combine_mem:>10.2f} MB")
print(f"  Embedding Fusion (peak):     {combine_peak_mem:>10.2f} MB\n")

print(f"  LLM Prefill (net):           {prefill_mem:>10.2f} MB")
print(f"  LLM Prefill (peak):          {prefill_peak_mem:>10.2f} MB\n")

print(f"  Initial KV-Cache:            {initial_kv_mem:>10.2f} MB")
print()

avg_decode_growth = sum(d['growth'] for d in decode_mems) / len(decode_mems)
print(f"KV-Cache Growth Analysis:")
print(f"  Average per token:           {avg_decode_growth:>10.4f} MB/token")
print(f"  Total growth (25 tokens):    {decode_mems[-1]['kv_cache'] - initial_kv_mem:>10.2f} MB")
print(f"  Final KV-Cache size:         {decode_mems[-1]['kv_cache']:>10.2f} MB\n")

total_inference_net = (transfer_mem + vision_mem + projector_mem + 
                       text_embed_mem + combine_mem + prefill_mem)

peak_phases = [
    transfer_mem,
    vision_peak_mem,
    projector_peak_mem,
    text_embed_peak_mem,
    combine_peak_mem,
    prefill_peak_mem
]
total_inference_peak = max(peak_phases)

print(f"Total Inference (net retained): {total_inference_net:>10.2f} MB\n")
print(f"Highest Single phase peak:         {total_inference_peak:>10.2f} MB\n")
print(f"Peak GPU Memory:                {decode_mems[-1]['total']:>10.2f} MB\n")

Static Components:
  Model Weights (FP16):          13472.42 MB

Dynamic Components (per inference):
  CPU→GPU Transfer:                  0.66 MB

  Vision Encoder (net):             10.25 MB
  Vision Encoder (peak):            53.25 MB

  Projection Layer (net):            4.51 MB
  Projection Layer (peak):           9.02 MB

  Text Embedding (net):              4.63 MB
  Text Embedding (peak):             4.63 MB

  Embedding Fusion (net):           10.02 MB
  Embedding Fusion (peak):          10.02 MB

  LLM Prefill (net):               338.99 MB
  LLM Prefill (peak):              359.11 MB

  Initial KV-Cache:                296.50 MB

KV-Cache Growth Analysis:
  Average per token:               0.5000 MB/token
  Total growth (25 tokens):         12.50 MB
  Final KV-Cache size:             309.00 MB

Total Inference (net retained):     369.06 MB

Highest Single phase peak:             359.11 MB

Peak GPU Memory:                  13819.12 MB



In [21]:
import json
from datetime import datetime

vram_profile = {
    "metadata": {
        "timestamp": datetime.utcnow().isoformat() + "Z",
        "precision": "FP16",
        "units": "MB"
    },

    "static_components": {
        "model_weights": model_mem
    },

    "dynamic_components": {
        "cpu_to_gpu_transfer": transfer_mem,

        "vision_encoder": {
            "net": vision_mem,
            "peak": vision_peak_mem
        },

        "projection_layer": {
            "net": projector_mem,
            "peak": projector_peak_mem
        },

        "text_embedding": {
            "net": text_embed_mem,
            "peak": text_embed_peak_mem
        },

        "embedding_fusion": {
            "net": combine_mem,
            "peak": combine_peak_mem
        },

        "llm_prefill": {
            "net": prefill_mem,
            "peak": prefill_peak_mem
        }
    },

    "kv_cache": {
        "initial": initial_kv_mem,
        "final": decode_mems[-1]["kv_cache"],
        "avg_growth_per_token": avg_decode_growth,
        "total_growth_25_tokens": decode_mems[-1]["kv_cache"] - initial_kv_mem,
        "per_token_trace": decode_mems
    },

    "aggregates": {
        "total_inference_net_retained": total_inference_net,
        "highest_single_phase_peak": total_inference_peak,
        "final_peak_gpu_memory": decode_mems[-1]["total"]
    }
}

with open("../metrics/vram_profile.json", "w") as f:
    json.dump(vram_profile, f, indent=2)
